# 02 — Features y protocolo temporal congelado

In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/sinteticos.py").is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from IPython.display import display, Image
from src.experimentos import seed_everything, holdout_indices, cv_indices, evaluate_cv, tune_grid


## Disponibilidad al emitir
`LOG_IMPORTE_EMISION` responde a la cola observada en EDA; `LOG_AREA_M2` transforma el area del registro de emision; seno/coseno del mes representan el calendario ya conocido. Plazo, zona ficticia, uso y canal tambien existen al emitir. Ninguna feature usa pagos, saldos o conteos futuros.

La etiqueta es saldo neto positivo al cierre del dia 90. Eventos administrativos de importe cero no extinguen deuda. Las etiquetas inmaduras no son clase 1; se excluyen.

In [2]:
from src.analisis import prepare_features
summary = prepare_features()
print(json.dumps(summary, indent=2))
from src.configuracion import load_experiment_contract
contract = load_experiment_contract()
print(json.dumps(contract, indent=2))

{
  "features": [
    "LOG_IMPORTE_EMISION",
    "LOG_AREA_M2",
    "PLAZO_DIAS_SIMULADO",
    "MES_EMISION_SIN",
    "MES_EMISION_COS",
    "ZONA_SIMULADA",
    "USO_PREDIO",
    "CANAL_NOTIFICACION"
  ],
  "train_rows": 9510,
  "holdout_rows": 2000,
  "embargo_rows": 490,
  "folds": [
    {
      "train": 3513,
      "validation": 2000
    },
    {
      "train": 5487,
      "validation": 2000
    },
    {
      "train": 7514,
      "validation": 1510
    }
  ],
  "holdout_evaluated": false,
  "cutoff": "2024-01-01",
  "final_observation_end": "2025-12-31"
}
{
  "approved": true,
  "approval_scope": "User-authorized first synthetic experimental version; academic assumptions",
  "dataset_version": "mc-eventos-v1",
  "code_version": "sha256:computed-at-run",
  "target": "TARGET_NO_PAGADA_90D",
  "target_rule_version": "net-cents-day90-inclusive-v1",
  "lote_ids": [
    "e614c2aa-fa87-553b-940f-9fdd6a5278a4"
  ],
  "numeric_features": [
    "LOG_IMPORTE_EMISION",
    "LOG_AREA_M2",
    

In [3]:
from src.ejecutar_estudio import leakage_diagnostics
print(json.dumps(leakage_diagnostics(), indent=2))

{
  "univariate_orientation_free_auc": {
    "LOG_IMPORTE_EMISION": 0.5035773409264314,
    "LOG_AREA_M2": 0.5050066000741755,
    "PLAZO_DIAS_SIMULADO": 0.5000694945812663,
    "MES_EMISION_SIN": 0.5560884447710757,
    "MES_EMISION_COS": 0.5074580208662708
  },
  "permuted_train_label_auc": [
    0.479664699776655,
    0.4713040077961216,
    0.48483239664221256
  ],
  "negative_control_auc_mean": 0.47860036807166306,
  "holdout_used": false,
  "excluded_future_columns": [
    "PAGADO_90_CENT",
    "SALDO_90_CENT",
    "N_MOVIMIENTOS",
    "LABEL_AVAILABLE_AT"
  ]
}


## Pipeline y hold-out
`build_pipeline` aplica FeatureGuard + ColumnTransformer con imputacion/escalado/OneHotEncoder dentro de cada train/CV. El hold-out 2024 no se evalua. Entrenamiento exige etiqueta disponible antes de 1-ene-2024; 490 filas tardias de 2023 quedan fuera por maduracion. CV expansiva valida 2021, 2022 y las 1.510 filas maduras elegibles de 2023. Las claves y folds se guardan en data/synthetic_v1/partitions.json. No se repiten obligaciones ni contribuyentes ficticios entre filas; no se asume un historial municipal.